# 🏋️ Day 05a: SQL Practice — 15 Interview-Grade Problems

---

## 🎯 What You'll Master Today
- 15 SQL problems from easy → hard
- Classic patterns: Top-N, consecutive, gaps, duplicates
- Each problem: SQL + Python simulation

---

In [ ]:
# ============================================
# Setup: Multi-table sample data
# ============================================

employees = [
    {"id": 1, "name": "Alice",   "dept": "Engineering", "salary": 95000, "mgr_id": None, "hire_date": "2020-01-15"},
    {"id": 2, "name": "Bob",     "dept": "Engineering", "salary": 85000, "mgr_id": 1,    "hire_date": "2020-06-01"},
    {"id": 3, "name": "Charlie", "dept": "Marketing",   "salary": 72000, "mgr_id": 1,    "hire_date": "2019-03-20"},
    {"id": 4, "name": "Diana",   "dept": "Marketing",   "salary": 68000, "mgr_id": 3,    "hire_date": "2021-07-10"},
    {"id": 5, "name": "Eve",     "dept": "Engineering", "salary": 92000, "mgr_id": 1,    "hire_date": "2019-11-05"},
    {"id": 6, "name": "Frank",   "dept": "Sales",       "salary": 65000, "mgr_id": None, "hire_date": "2018-08-22"},
    {"id": 7, "name": "Grace",   "dept": "Sales",       "salary": 71000, "mgr_id": 6,    "hire_date": "2022-01-03"},
    {"id": 8, "name": "Hank",    "dept": "Engineering", "salary": 98000, "mgr_id": 1,    "hire_date": "2017-05-14"},
    {"id": 9, "name": "Ivy",     "dept": "Marketing",   "salary": 72000, "mgr_id": 3,    "hire_date": "2023-02-28"},
]

orders = [
    {"id": 1, "emp_id": 6, "amount": 5000,  "date": "2024-01-10"},
    {"id": 2, "emp_id": 7, "amount": 3200,  "date": "2024-01-15"},
    {"id": 3, "emp_id": 6, "amount": 7500,  "date": "2024-02-01"},
    {"id": 4, "emp_id": 7, "amount": 4100,  "date": "2024-02-12"},
    {"id": 5, "emp_id": 6, "amount": 2800,  "date": "2024-03-05"},
    {"id": 6, "emp_id": 7, "amount": 9200,  "date": "2024-03-20"},
]

logins = [
    {"user_id": 1, "date": "2024-01-01"}, {"user_id": 1, "date": "2024-01-02"},
    {"user_id": 1, "date": "2024-01-03"}, {"user_id": 1, "date": "2024-01-05"},
    {"user_id": 1, "date": "2024-01-06"}, {"user_id": 2, "date": "2024-01-01"},
    {"user_id": 2, "date": "2024-01-03"}, {"user_id": 2, "date": "2024-01-04"},
]

def pt(rows, cols=None):
    if not rows: print("  (empty)"); return
    if cols is None: cols = list(rows[0].keys())
    w = {c: max(len(str(c)), max(len(str(r.get(c,''))) for r in rows)) for c in cols}
    print("  " + " | ".join(f"{c:<{w[c]}}" for c in cols))
    print("  " + "-+-".join("-"*w[c] for c in cols))
    for r in rows:
        print("  " + " | ".join(f"{str(r.get(c,'')):<{w[c]}}" for c in cols))

print("  Data loaded: employees(9), orders(6), logins(8)")

In [ ]:
# ============================================
# P1: Second highest salary
# ============================================
# SQL: SELECT MAX(salary) FROM employees WHERE salary < (SELECT MAX(salary) FROM employees);

print("  P1: Second highest salary")
max_sal = max(e["salary"] for e in employees)
second = max(e["salary"] for e in employees if e["salary"] < max_sal)
print(f"  Answer: {second}")
print("  Alt SQL: SELECT DISTINCT salary FROM employees ORDER BY salary DESC LIMIT 1 OFFSET 1;")
print("  Alt SQL: DENSE_RANK() OVER (ORDER BY salary DESC) = 2")

In [ ]:
# ============================================
# P2: Nth highest salary (generalized)
# ============================================

print("  P2: Nth highest salary (N=3)")
N = 3
unique_sals = sorted(set(e["salary"] for e in employees), reverse=True)
print(f"  Unique salaries desc: {unique_sals}")
print(f"  {N}rd highest: {unique_sals[N-1] if N <= len(unique_sals) else 'NULL'}")
print("  SQL: WITH ranked AS (SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS dr FROM employees)")
print("       SELECT salary FROM ranked WHERE dr = N;")

In [ ]:
# ============================================
# P3: Find duplicate records
# ============================================

print("  P3: Find duplicate salaries")
# SQL: SELECT salary, COUNT(*) FROM employees GROUP BY salary HAVING COUNT(*) > 1;
from collections import Counter
sal_counts = Counter(e["salary"] for e in employees)
dups = [{"salary": s, "count": c} for s, c in sal_counts.items() if c > 1]
pt(dups)
print("  SQL: GROUP BY salary HAVING COUNT(*) > 1")

In [ ]:
# ============================================
# P4: Employees earning more than their manager
# ============================================

print("  P4: Employees earning more than their manager")
# SQL: SELECT e.name FROM employees e JOIN employees m ON e.mgr_id = m.id WHERE e.salary > m.salary;

emp_lookup = {e["id"]: e for e in employees}
result = []
for e in employees:
    if e["mgr_id"] and e["mgr_id"] in emp_lookup:
        mgr = emp_lookup[e["mgr_id"]]
        if e["salary"] > mgr["salary"]:
            result.append({"employee": e["name"], "emp_sal": e["salary"],
                           "manager": mgr["name"], "mgr_sal": mgr["salary"]})
pt(result)
print("  SQL: SELF JOIN + WHERE e.salary > m.salary")

In [ ]:
# ============================================
# P5: Top 2 earners per department
# ============================================

print("  P5: Top 2 earners per department")
from collections import defaultdict
groups = defaultdict(list)
for e in employees:
    groups[e["dept"]].append(e)

result = []
for dept in sorted(groups):
    top2 = sorted(groups[dept], key=lambda x: x["salary"], reverse=True)[:2]
    for i, e in enumerate(top2):
        result.append({"dept": dept, "name": e["name"], "salary": e["salary"], "rank": i+1})
pt(result)
print("  SQL: ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC) + WHERE rn <= 2")

In [ ]:
# ============================================
# P6: Department with highest avg salary
# ============================================

print("  P6: Department with highest average salary")
dept_avgs = []
for dept, emps in groups.items():
    avg = sum(e["salary"] for e in emps) / len(emps)
    dept_avgs.append({"dept": dept, "avg_salary": round(avg), "count": len(emps)})
dept_avgs.sort(key=lambda x: x["avg_salary"], reverse=True)
pt(dept_avgs)
print(f"  Winner: {dept_avgs[0]['dept']}")
print("  SQL: SELECT dept, AVG(salary) FROM employees GROUP BY dept ORDER BY AVG(salary) DESC LIMIT 1;")

In [ ]:
# ============================================
# P7: Consecutive login days (gap-and-island)
# ============================================

print("  P7: Find max consecutive login days per user\n")
from datetime import datetime, timedelta

user_logins = defaultdict(list)
for l in logins:
    user_logins[l["user_id"]].append(datetime.strptime(l["date"], "%Y-%m-%d"))

# Gap-and-island technique: date - row_number = same for consecutive
for uid, dates in user_logins.items():
    dates.sort()
    # Assign group: date - index gives same value for consecutive dates
    islands = []
    for i, d in enumerate(dates):
        group_key = d - timedelta(days=i)
        islands.append({"date": d.strftime("%Y-%m-%d"), "group": group_key.strftime("%Y-%m-%d")})
    
    # Count consecutive streaks
    group_counts = Counter(i["group"] for i in islands)
    max_streak = max(group_counts.values())
    print(f"  User {uid}: dates={[d.strftime('%m-%d') for d in dates]}")
    print(f"    Groups: {[(i['date'], i['group']) for i in islands]}")
    print(f"    Max consecutive days: {max_streak}\n")

print("  SQL Technique: date - ROW_NUMBER() OVER (ORDER BY date) AS grp")
print("  Then GROUP BY grp, COUNT(*) gives streak lengths")

In [ ]:
# ============================================
# P8: Running total of orders per salesperson
# ============================================

print("  P8: Running total of orders per salesperson\n")

# SQL: SELECT emp_id, date, amount,
#        SUM(amount) OVER (PARTITION BY emp_id ORDER BY date) AS running_total
#      FROM orders;

order_groups = defaultdict(list)
for o in orders:
    order_groups[o["emp_id"]].append(o)

result = []
for eid in sorted(order_groups):
    running = 0
    for o in sorted(order_groups[eid], key=lambda x: x["date"]):
        running += o["amount"]
        result.append({"emp_id": eid, "date": o["date"], "amount": o["amount"], "running_total": running})

pt(result)
print("  SQL: SUM(amount) OVER (PARTITION BY emp_id ORDER BY date)")

In [ ]:
# ============================================
# P9-P12: Quick-fire problems
# ============================================

print("  P9: Delete duplicate rows (keep lowest id)")
print("  SQL: DELETE FROM employees WHERE id NOT IN")
print("       (SELECT MIN(id) FROM employees GROUP BY name, salary);\n")

print("  P10: Employees hired in last N months")
print("  SQL: WHERE hire_date >= DATE_SUB(CURDATE(), INTERVAL 6 MONTH);\n")

print("  P11: Median salary")
sals = sorted(e["salary"] for e in employees)
n = len(sals)
median = sals[n//2] if n % 2 else (sals[n//2-1] + sals[n//2]) / 2
print(f"  Sorted: {sals}")
print(f"  Median: {median}")
print("  SQL: Use PERCENTILE_CONT(0.5) or ROW_NUMBER trick\n")

print("  P12: Departments with no employees")
print("  SQL: SELECT d.name FROM departments d")
print("       LEFT JOIN employees e ON d.id = e.dept_id")
print("       WHERE e.id IS NULL;  (anti-join pattern)")

In [ ]:
# ============================================
# P13-P15: Advanced problems
# ============================================

print("  P13: Year-over-year growth")
print("  SQL: WITH yearly AS (SELECT YEAR(date) yr, SUM(amount) total FROM orders GROUP BY yr)")
print("       SELECT yr, total, LAG(total) OVER (ORDER BY yr) prev,")
print("       ROUND((total - LAG(total) OVER (ORDER BY yr)) / LAG(total) OVER (ORDER BY yr) * 100, 1) growth")
print("       FROM yearly;\n")

print("  P14: Pivot monthly sales (rows → columns)")
print("  SQL: SELECT emp_id,")
print("         SUM(CASE WHEN MONTH(date)=1 THEN amount END) AS Jan,")
print("         SUM(CASE WHEN MONTH(date)=2 THEN amount END) AS Feb,")
print("         SUM(CASE WHEN MONTH(date)=3 THEN amount END) AS Mar")
print("       FROM orders GROUP BY emp_id;\n")

# Simulate pivot
pivot = defaultdict(lambda: defaultdict(int))
for o in orders:
    month = o["date"][5:7]
    pivot[o["emp_id"]][month] += o["amount"]

result = [{"emp_id": eid, "Jan": ms.get("01",0), "Feb": ms.get("02",0), "Mar": ms.get("03",0)}
          for eid, ms in sorted(pivot.items())]
pt(result)

print("\n  P15: Find gaps in sequential IDs")
ids = [1,2,3,5,6,9,10]
gaps = []
for i in range(len(ids)-1):
    if ids[i+1] - ids[i] > 1:
        gaps.append({"gap_start": ids[i]+1, "gap_end": ids[i+1]-1})
print(f"  IDs: {ids}")
pt(gaps)
print("  SQL: SELECT id+1 AS gap_start, LEAD(id) OVER (ORDER BY id)-1 AS gap_end")
print("       FROM my_table WHERE LEAD(id) OVER (ORDER BY id) - id > 1;")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | How to find Nth highest? | DENSE_RANK or LIMIT/OFFSET or correlated subquery |
| 2 | How to delete duplicates? | DELETE WHERE id NOT IN (SELECT MIN(id) GROUP BY dup_cols) |
| 3 | Gap-and-island technique? | date - ROW_NUMBER() = same constant for consecutive dates |
| 4 | How to pivot rows to columns? | CASE WHEN + GROUP BY + aggregate |
| 5 | Anti-join pattern? | LEFT JOIN + WHERE right_key IS NULL |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Nth highest: DENSE_RANK > subquery > LIMIT OFFSET    │
## │  • Consecutive: gap-and-island (date - row_number)      │
## │  • Duplicates: GROUP BY + HAVING COUNT > 1              │
## │  • Pivot: CASE WHEN inside aggregate + GROUP BY         │
## │  • Practice these 15 patterns — they cover 80% of Qs   │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **ACID Properties** — Atomicity, Consistency, Isolation, Durability